# 📬 Pulso Académico SIIMAG — v2 (Infografía)

Boletín trimestral con diseño editorial oscuro. Fondo negro, tipografía bold, íconos SVG geométricos y bloques de color por categoría.

**Compatible con:** Gmail (Google Workspace) ✅

---
**Flujo del notebook:**
1. Setup e instalación
2. Conexión a BD
3. Extracción de indicadores
4. Texto introductorio con Gemini
5. Generación del HTML
6. Preview en Colab
7. Envío por correo

In [ ]:
# ── 1. SETUP ────────────────────────────────────────────────────────────────
!pip install git+https://github.com/claudiodanielpc-ag/cd_base.git -q
!pip install unidecode -q

from cd_base import ConexionBD
from google.colab import drive, ai
drive.mount('/content/drive')

import pandas as pd
import re, unidecode, smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText

print('✅ Librerías listas')

In [ ]:
# ── 2. CONEXIÓN A BD ─────────────────────────────────────────────────────────
ruta  = '/content/drive/MyDrive/credenciales/bd_produccion.txt'
bd    = ConexionBD(ruta)
engine = bd.conectar('migracion_aws_do')

In [ ]:
# ── 3. PARÁMETROS DEL PERIODO ─────────────────────────────────────────────────
# Ajusta estas fechas cada trimestre
PERIODO_ACTUAL_INI = '2026-01-01'
PERIODO_ACTUAL_FIN = '2026-03-31'
PERIODO_COMP_INI   = '2025-01-01'
PERIODO_COMP_FIN   = '2025-03-31'
PERIODO_LABEL      = 'Enero – Marzo 2026'
PERIODO_COMP_LABEL = 'Enero – Marzo 2025'
PERIODO_CORTO      = 'ENE–MAR 2026'   # Para el encabezado

In [ ]:
# ── 4. EXTRACCIÓN DE INDICADORES ─────────────────────────────────────────────
conn   = engine.raw_connection()
cursor = conn.cursor()

cursor.callproc(
    'sp_tbl_indicadores_general_programas',
    (1, None, None,
     PERIODO_ACTUAL_INI, PERIODO_ACTUAL_FIN,
     PERIODO_COMP_INI,   PERIODO_COMP_FIN)
)

resultados = []
while True:
    rows = cursor.fetchall()
    if rows:
        cols = [col[0] for col in cursor.description]
        resultados.append(pd.DataFrame(rows, columns=cols))
    if not cursor.nextset():
        break
cursor.close()

indicadores = resultados[0]
print(f'✅ {len(indicadores)} indicadores cargados')
indicadores

In [ ]:
# ── 4-ALT. DATOS DE EJEMPLO (descomenta si no tienes BD) ─────────────────────

# indicadores = pd.DataFrame([
#   {'id_indicador':3,  'nombre':'Inscritos',            'porcentual':0,'categoria':1,'total':570,  'total_comp':641,   'porcentaje':-11.08,'tendencia':'down','descripcion':'Alumnos dados de alta en el programa académico.'},
#   {'id_indicador':4,  'nombre':'Inscritos facturados', 'porcentual':0,'categoria':1,'total':558,  'total_comp':542,   'porcentaje': 2.95, 'tendencia':'up',  'descripcion':'Inscritos con al menos una materia cargada.'},
#   {'id_indicador':6,  'nombre':'Cargas',               'porcentual':0,'categoria':1,'total':15854,'total_comp':16854, 'porcentaje':-5.93, 'tendencia':'down','descripcion':'Materias inscritas durante el periodo.'},
#   {'id_indicador':9,  'nombre':'Eficiencia terminal',  'porcentual':1,'categoria':1,'total':30.37,'total_comp':25.51, 'porcentaje':19.05, 'tendencia':'up',  'descripcion':'Porcentaje de alumnos que concluyeron materias.'},
#   {'id_indicador':5,  'nombre':'Reactivados',          'porcentual':0,'categoria':2,'total':137,  'total_comp':321,   'porcentaje':-57.32,'tendencia':'down','descripcion':'Alumnos que retomaron su trayectoria.'},
#   {'id_indicador':12, 'nombre':'Tasa de reactivación', 'porcentual':1,'categoria':2,'total':63.72,'total_comp':126.88,'porcentaje':-49.78,'tendencia':'down','descripcion':'Porcentaje de alumnos que retoman vs. bajas.'},
#   {'id_indicador':10, 'nombre':'Reinscritos',          'porcentual':0,'categoria':2,'total':158,  'total_comp':160,   'porcentaje':-1.25, 'tendencia':'down','descripcion':'Alumnos que retomaron tras baja de empresa.'},
#   {'id_indicador':7,  'nombre':'Tasa de reinserción',  'porcentual':1,'categoria':2,'total':31.35,'total_comp':30.36, 'porcentaje': 3.26, 'tendencia':'up',  'descripcion':'Porcentaje que vuelve a inscribirse.'},
#   {'id_indicador':1,  'nombre':'Bajas de la empresa',  'porcentual':0,'categoria':3,'total':504,  'total_comp':527,   'porcentaje':-4.36, 'tendencia':'down','descripcion':'Alumnos que dejaron de laborar en la empresa.'},
#   {'id_indicador':2,  'nombre':'Bajas del programa',   'porcentual':0,'categoria':3,'total':202,  'total_comp':187,   'porcentaje': 8.02, 'tendencia':'up',  'descripcion':'Alumnos que dejaron el programa pero siguen en empresa.'},
#   {'id_indicador':8,  'nombre':'Tasa de Deserción',    'porcentual':1,'categoria':3,'total':36.20,'total_comp':34.50, 'porcentaje': 4.93, 'tendencia':'up',  'descripcion':'Porcentaje que no continúa el programa.'},
# ])
# print('✅ Datos de ejemplo listos')

In [ ]:
# ── 5. TEXTO INTRODUCTORIO CON GEMINI ────────────────────────────────────────
resumen = indicadores[['nombre','total','total_comp','porcentaje','tendencia']].to_string(index=False)

prompt = f"""
Eres el analista de datos del SIIMAG (Academia Global).
Escribe UN párrafo de máximo 2 oraciones para el boletín trimestral {PERIODO_LABEL}.
Tono: directo, ejecutivo. Menciona 2 indicadores clave con sus valores numéricos.
No uses listas, solo prosa. Sin título ni encabezado. Sin asteriscos ni markdown.

Datos:
{resumen}
"""

texto_intro = ai.generate_text(prompt).strip()
print(texto_intro)

In [ ]:
# ── 6. FUNCIONES AUXILIARES ───────────────────────────────────────────────────

CATEGORIAS = {1: 'Captación', 2: 'Retención', 3: 'Bajas y deserción'}

# Colores del tema oscuro
COLOR_FONDO      = '#0A0A0A'
COLOR_BLOQUE     = '#111111'
COLOR_SEPARADOR  = '#1e1e1e'
COLOR_TEXTO      = '#F2EFE6'
COLOR_MUTED      = '#555555'
COLOR_ACENTO     = '#E8C14A'  # Dorado
COLOR_VERDE      = '#4ade80'
COLOR_ROJO       = '#f87171'
COLOR_GRIS       = '#888888'

def fmt_valor(row):
    v = row['total']
    if row['porcentual'] == 1:
        return f"{v:,.1f}", '%'
    elif v >= 1000:
        return f"{int(v):,}", ''
    else:
        return f"{int(v)}", ''

def fmt_comp(row):
    v = row['total_comp']
    if row['porcentual'] == 1:
        return f"{v:,.1f}%"
    elif v >= 1000:
        return f"{int(v):,}"
    return f"{int(v)}"

def es_bueno(row):
    """True si el cambio es positivo para el programa."""
    sube = row['porcentaje'] > 0
    return (sube and row['tendencia'] == 'up') or (not sube and row['tendencia'] == 'down')

def color_var(row):
    if row['porcentaje'] == 0: return COLOR_GRIS
    return COLOR_VERDE if es_bueno(row) else COLOR_ROJO

def bg_var(row):
    if row['porcentaje'] == 0: return '#1a1a1a'
    return '#0d2a12' if es_bueno(row) else '#2a0d0d'

def flecha(row):
    return '&#9650;' if row['porcentaje'] > 0 else '&#9660;'

def color_barra(row):
    if row['porcentaje'] == 0: return COLOR_GRIS
    return COLOR_VERDE if es_bueno(row) else COLOR_ROJO

print('✅ Funciones listas')

In [ ]:
# ── 7. ÍCONOS SVG POR INDICADOR ───────────────────────────────────────────────
# Mapa de ícono geométrico por id_indicador
ICONOS = {
    3:  '''<svg width="22" height="22" viewBox="0 0 24 24" fill="none">
             <rect x="2" y="2" width="8" height="8" rx="1" fill="#E8C14A" opacity="0.9"/>
             <rect x="14" y="2" width="8" height="8" rx="1" fill="#E8C14A" opacity="0.5"/>
             <rect x="2" y="14" width="8" height="8" rx="1" fill="#E8C14A" opacity="0.5"/>
             <rect x="14" y="14" width="8" height="8" rx="1" fill="#E8C14A" opacity="0.2"/>
           </svg>''',
    4:  '''<svg width="22" height="22" viewBox="0 0 24 24" fill="none">
             <circle cx="12" cy="12" r="10" stroke="#4ade80" stroke-width="1.5" opacity="0.5"/>
             <circle cx="12" cy="12" r="6" fill="#4ade80" opacity="0.7"/>
           </svg>''',
    6:  '''<svg width="22" height="22" viewBox="0 0 24 24" fill="none">
             <rect x="2" y="14" width="4" height="8" rx="1" fill="#E8C14A" opacity="0.9"/>
             <rect x="8" y="9"  width="4" height="13" rx="1" fill="#E8C14A" opacity="0.6"/>
             <rect x="14" y="5" width="4" height="17" rx="1" fill="#E8C14A" opacity="0.4"/>
             <rect x="20" y="2" width="4" height="20" rx="1" fill="#E8C14A" opacity="0.2"/>
           </svg>''',
    9:  '''<svg width="22" height="22" viewBox="0 0 24 24" fill="none">
             <polygon points="12,2 22,22 2,22" stroke="#4ade80" stroke-width="1.5" fill="rgba(74,222,128,0.1)"/>
             <line x1="12" y1="10" x2="12" y2="18" stroke="#4ade80" stroke-width="1.5"/>
           </svg>''',
    5:  '''<svg width="22" height="22" viewBox="0 0 24 24" fill="none">
             <path d="M12 2 L22 12 L12 22 L2 12 Z" stroke="#f87171" stroke-width="1.5" fill="rgba(248,113,113,0.1)"/>
             <circle cx="12" cy="12" r="2" fill="#f87171"/>
           </svg>''',
    12: '''<svg width="22" height="22" viewBox="0 0 24 24" fill="none">
             <circle cx="12" cy="12" r="10" stroke="#f87171" stroke-width="1.5" opacity="0.4"/>
             <path d="M6 12 Q12 6 18 12" stroke="#f87171" stroke-width="1.5" fill="none" stroke-linecap="round"/>
           </svg>''',
    10: '''<svg width="22" height="22" viewBox="0 0 24 24" fill="none">
             <rect x="2" y="2" width="20" height="20" rx="3" stroke="#888" stroke-width="1.5" fill="none"/>
             <line x1="7" y1="12" x2="17" y2="12" stroke="#888" stroke-width="1.5"/>
           </svg>''',
    7:  '''<svg width="22" height="22" viewBox="0 0 24 24" fill="none">
             <path d="M4 20 C4 12 20 12 20 4" stroke="#4ade80" stroke-width="2" stroke-linecap="round" fill="none" opacity="0.7"/>
             <circle cx="20" cy="4" r="3" fill="#4ade80" opacity="0.9"/>
           </svg>''',
    1:  '''<svg width="22" height="22" viewBox="0 0 24 24" fill="none">
             <path d="M12 3 L3 21 L21 21 Z" stroke="#4ade80" stroke-width="1.5" stroke-linejoin="round" fill="rgba(74,222,128,0.1)"/>
           </svg>''',
    2:  '''<svg width="22" height="22" viewBox="0 0 24 24" fill="none">
             <path d="M12 21 L3 3 L21 3 Z" stroke="#f87171" stroke-width="1.5" stroke-linejoin="round" fill="rgba(248,113,113,0.1)"/>
           </svg>''',
    8:  '''<svg width="22" height="22" viewBox="0 0 24 24" fill="none">
             <path d="M4 20 C4 12 20 12 20 4" stroke="#f87171" stroke-width="2" stroke-linecap="round" fill="none"/>
             <circle cx="20" cy="4" r="3" fill="#f87171"/>
           </svg>''',
}

DEFAULT_ICONO = '''<svg width="22" height="22" viewBox="0 0 24 24" fill="none">
  <circle cx="12" cy="12" r="9" stroke="#555" stroke-width="1.5"/>
</svg>'''

print('✅ Íconos definidos')

In [ ]:
# ── 8. GENERADORES DE BLOQUES HTML ────────────────────────────────────────────

CSS_FUENTE = "font-family:'Syne',Arial,sans-serif;"
CSS_MONO   = "font-family:'Syne Mono','Courier New',monospace;"

def bloque_kpi_grande(row):
    """Bloque grande para la sección Captación (3 columnas)."""
    valor, sufijo = fmt_valor(row)
    comp    = fmt_comp(row)
    cv      = color_var(row)
    bv      = bg_var(row)
    fl      = flecha(row)
    pct     = abs(row['porcentaje'])
    icono   = ICONOS.get(row['id_indicador'], DEFAULT_ICONO)
    sufijo_html = f'<span style="font-size:15px;color:{COLOR_MUTED};">{sufijo}</span>' if sufijo else ''

    return f"""
    <td width="33%" style="padding:20px 18px;vertical-align:top;background:{COLOR_BLOQUE};">
      <div style="margin-bottom:14px;">{icono}</div>
      <div style="font-size:32px;font-weight:800;color:{COLOR_TEXTO};line-height:1;margin-bottom:4px;{CSS_FUENTE}">
        {valor}{sufijo_html}
      </div>
      <div style="font-size:10px;letter-spacing:1.5px;color:{COLOR_MUTED};text-transform:uppercase;
                  margin-bottom:10px;{CSS_FUENTE}">{row['nombre']}</div>
      <span style="display:inline-block;background:{bv};color:{cv};
                   padding:3px 8px;border-radius:4px;font-size:11px;{CSS_MONO}">
        {fl} {pct:.1f}%
      </span>
    </td>"""


def bloque_kpi_horizontal(row):
    """Bloque horizontal para Retención y Bajas (2 columnas)."""
    valor, sufijo = fmt_valor(row)
    comp    = fmt_comp(row)
    cv      = color_var(row)
    cb      = color_barra(row)
    fl      = flecha(row)
    pct     = abs(row['porcentaje'])
    sufijo_html = f'<span style="font-size:12px;color:{COLOR_MUTED};">{sufijo}</span>' if sufijo else ''

    return f"""
    <td width="50%" style="padding:18px;vertical-align:top;background:{COLOR_BLOQUE};">
      <table cellpadding="0" cellspacing="0" width="100%">
        <tr>
          <td width="4" style="background:{cb};border-radius:2px;width:3px;">&nbsp;</td>
          <td style="padding-left:14px;">
            <div style="font-size:24px;font-weight:800;color:{COLOR_TEXTO};line-height:1;
                        margin-bottom:3px;{CSS_FUENTE}">{valor}{sufijo_html}</div>
            <div style="font-size:10px;letter-spacing:1px;color:{COLOR_MUTED};
                        text-transform:uppercase;margin-bottom:8px;{CSS_FUENTE}">{row['nombre']}</div>
            <div style="color:{cv};font-size:11px;{CSS_MONO}">{fl} {pct:.1f}%</div>
            <div style="font-size:10px;color:#444;margin-top:3px;{CSS_FUENTE}">vs. {comp} en {PERIODO_COMP_LABEL[-4:]}</div>
          </td>
        </tr>
      </table>
    </td>"""


def etiqueta_seccion(label):
    return f"""
    <tr>
      <td colspan="99" style="padding:18px 28px 6px;background:{COLOR_FONDO};">
        <table cellpadding="0" cellspacing="0" width="100%">
          <tr>
            <td style="font-size:9px;letter-spacing:3px;color:#444;
                       text-transform:uppercase;padding-right:12px;
                       white-space:nowrap;{CSS_FUENTE}">{label}</td>
            <td style="border-top:1px solid {COLOR_SEPARADOR};width:100%;"></td>
          </tr>
        </table>
      </td>
    </tr>"""


print('✅ Generadores de bloques listos')

In [ ]:
# ── 9. CONSTRUCCIÓN DEL HTML ──────────────────────────────────────────────────

# -- Captación (3 columnas) --
cat1 = indicadores[indicadores['categoria'] == 1].reset_index(drop=True)
celdas_cap = ''.join(bloque_kpi_grande(row) for _, row in cat1.iterrows())
filas_captacion = f"""
<tr style="gap:1px;">{celdas_cap}</tr>
"""

# -- Retención (2 columnas, filas de a 2) --
cat2 = indicadores[indicadores['categoria'] == 2].reset_index(drop=True)
filas_retencion = ''
for i in range(0, len(cat2), 2):
    par = cat2.iloc[i:i+2]
    celdas = ''.join(bloque_kpi_horizontal(row) for _, row in par.iterrows())
    if len(par) == 1:
        celdas += f'<td width="50%" style="background:{COLOR_BLOQUE};"></td>'
    filas_retencion += f'<tr style="gap:1px;">{celdas}</tr>'

# -- Bajas (2 columnas, filas de a 2) --
cat3 = indicadores[indicadores['categoria'] == 3].reset_index(drop=True)
filas_bajas = ''
for i in range(0, len(cat3), 2):
    par = cat3.iloc[i:i+2]
    celdas = ''.join(bloque_kpi_horizontal(row) for _, row in par.iterrows())
    if len(par) == 1:
        celdas += f'<td width="50%" style="background:{COLOR_BLOQUE};"></td>'
    filas_bajas += f'<tr style="gap:1px;">{celdas}</tr>'


# -- HTML completo --
html = f"""<!DOCTYPE html>
<html lang="es">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width,initial-scale=1.0">
  <title>Pulso Académico SIIMAG · {PERIODO_LABEL}</title>
  <link href="https://fonts.googleapis.com/css2?family=Syne:wght@400;700;800&family=Syne+Mono&display=swap" rel="stylesheet">
</head>
<body style="margin:0;padding:0;background:#f0f0f0;{CSS_FUENTE}">

<table width="100%" cellpadding="0" cellspacing="0" style="background:#f0f0f0;padding:24px 0;">
<tr><td align="center">

<table width="620" cellpadding="0" cellspacing="0"
  style="background:{COLOR_FONDO};max-width:620px;
         border-radius:16px;overflow:hidden;border:1px solid #1e1e1e;">

  <!-- BARRA SUPERIOR -->
  <tr>
    <td colspan="99" style="background:{COLOR_FONDO};
        border-bottom:1px solid {COLOR_SEPARADOR};padding:16px 28px 14px;">
      <table width="100%" cellpadding="0" cellspacing="0">
        <tr>
          <td>
            <table cellpadding="0" cellspacing="0">
              <tr>
                <td width="10" height="10" style="background:{COLOR_ACENTO};
                    border-radius:50%;">&nbsp;</td>
                <td style="padding-left:10px;font-size:11px;letter-spacing:3px;
                           color:#888;text-transform:uppercase;{CSS_FUENTE}">
                  SIIMAG · Academia Global
                </td>
              </tr>
            </table>
          </td>
          <td align="right">
            <span style="background:#1a1a1a;border:1px solid #2a2a2a;
                         padding:5px 14px;border-radius:20px;
                         font-size:10px;letter-spacing:1.5px;color:#666;
                         {CSS_MONO}">{PERIODO_CORTO}</span>
          </td>
        </tr>
      </table>
    </td>
  </tr>

  <!-- HERO -->
  <tr>
    <td colspan="99" style="padding:32px 28px 24px;background:{COLOR_FONDO};">
      <table width="100%" cellpadding="0" cellspacing="0">
        <tr>
          <td width="55%" style="vertical-align:top;padding-right:20px;">
            <div style="font-size:10px;letter-spacing:3px;color:{COLOR_ACENTO};
                        text-transform:uppercase;margin-bottom:12px;{CSS_FUENTE}">
              Pulso Académico
            </div>
            <div style="font-size:42px;font-weight:800;line-height:1.0;
                        color:{COLOR_TEXTO};margin-bottom:16px;{CSS_FUENTE}">
              {len(indicadores)}<br>indica<span style="color:{COLOR_ACENTO}">dores</span><br>clave
            </div>
            <div style="font-size:13px;color:#666;line-height:1.6;{CSS_FUENTE}">
              {texto_intro}
            </div>
          </td>
          <td width="45%" align="center" style="vertical-align:middle;">
            <!-- Anillos SVG de categorías -->
            <div style="position:relative;display:inline-block;width:170px;height:170px;">
              <svg width="170" height="170" viewBox="0 0 170 170">
                <circle cx="85" cy="85" r="74" stroke="{COLOR_SEPARADOR}" stroke-width="11" fill="none"/>
                <circle cx="85" cy="85" r="74" stroke="{COLOR_ACENTO}" stroke-width="11"
                  stroke-dasharray="232 233" stroke-dashoffset="116"
                  stroke-linecap="round" fill="none"/>
                <circle cx="85" cy="85" r="55" stroke="{COLOR_SEPARADOR}" stroke-width="8" fill="none"/>
                <circle cx="85" cy="85" r="55" stroke="{COLOR_VERDE}" stroke-width="8"
                  stroke-dasharray="173 174" stroke-dashoffset="43"
                  stroke-linecap="round" fill="none"/>
                <circle cx="85" cy="85" r="38" stroke="{COLOR_SEPARADOR}" stroke-width="6" fill="none"/>
                <circle cx="85" cy="85" r="38" stroke="{COLOR_ROJO}" stroke-width="6"
                  stroke-dasharray="119 120" stroke-dashoffset="-60"
                  stroke-linecap="round" fill="none"/>
                <text x="85" y="80" text-anchor="middle" fill="{COLOR_TEXTO}"
                  font-family="Syne, Arial, sans-serif" font-size="26" font-weight="800">3</text>
                <text x="85" y="98" text-anchor="middle" fill="#555"
                  font-family="Syne, Arial, sans-serif" font-size="9" letter-spacing="2">CATEGORÍAS</text>
              </svg>
            </div>
            <!-- Leyenda de anillos -->
            <table cellpadding="0" cellspacing="0" style="margin-top:12px;">
              <tr>
                <td style="padding:2px 8px 2px 0;">
                  <span style="display:inline-block;width:10px;height:3px;
                               background:{COLOR_ACENTO};border-radius:2px;
                               vertical-align:middle;"></span>
                  <span style="font-size:9px;color:#555;letter-spacing:1px;
                               {CSS_FUENTE};margin-left:5px;">Captación</span>
                </td>
              </tr>
              <tr>
                <td style="padding:2px 8px 2px 0;">
                  <span style="display:inline-block;width:10px;height:3px;
                               background:{COLOR_VERDE};border-radius:2px;
                               vertical-align:middle;"></span>
                  <span style="font-size:9px;color:#555;letter-spacing:1px;
                               {CSS_FUENTE};margin-left:5px;">Retención</span>
                </td>
              </tr>
              <tr>
                <td style="padding:2px 0 2px 0;">
                  <span style="display:inline-block;width:10px;height:3px;
                               background:{COLOR_ROJO};border-radius:2px;
                               vertical-align:middle;"></span>
                  <span style="font-size:9px;color:#555;letter-spacing:1px;
                               {CSS_FUENTE};margin-left:5px;">Bajas</span>
                </td>
              </tr>
            </table>
          </td>
        </tr>
      </table>
    </td>
  </tr>

  <!-- DIVISOR -->
  <tr>
    <td colspan="99" style="height:1px;background:{COLOR_SEPARADOR};"></td>
  </tr>

  <!-- CAPTACIÓN -->
  {etiqueta_seccion('Captación')}
  <tr>
    <td colspan="99" style="padding:0 28px 8px;background:{COLOR_FONDO};">
      <table width="100%" cellpadding="0" cellspacing="1"
        style="background:{COLOR_SEPARADOR};border-radius:10px;overflow:hidden;">
        {filas_captacion}
      </table>
    </td>
  </tr>

  <!-- RETENCIÓN -->
  {etiqueta_seccion('Retención')}
  <tr>
    <td colspan="99" style="padding:0 28px 8px;background:{COLOR_FONDO};">
      <table width="100%" cellpadding="0" cellspacing="1"
        style="background:{COLOR_SEPARADOR};border-radius:10px;overflow:hidden;">
        {filas_retencion}
      </table>
    </td>
  </tr>

  <!-- BAJAS -->
  {etiqueta_seccion('Bajas y deserción')}
  <tr>
    <td colspan="99" style="padding:0 28px 24px;background:{COLOR_FONDO};">
      <table width="100%" cellpadding="0" cellspacing="1"
        style="background:{COLOR_SEPARADOR};border-radius:10px;overflow:hidden;">
        {filas_bajas}
      </table>
    </td>
  </tr>

  <!-- FRANJA CTA DORADA -->
  <tr>
    <td colspan="99" style="background:{COLOR_ACENTO};padding:14px 28px;">
      <table width="100%" cellpadding="0" cellspacing="0">
        <tr>
          <td style="font-size:11px;color:{COLOR_FONDO};font-weight:700;
                     letter-spacing:0.5px;{CSS_FUENTE}">
            Sistema de Información, Inteligencia y Monitoreo · Academia Global
          </td>
          <td align="right">
            <a href="https://erp.agcollege.com.mx/#/login"
               style="font-size:10px;color:{COLOR_FONDO};letter-spacing:2px;
                      text-transform:uppercase;text-decoration:none;
                      font-weight:400;opacity:0.7;{CSS_FUENTE}">
              Ir al SIIMAG →
            </a>
          </td>
        </tr>
      </table>
    </td>
  </tr>

</table>
</td></tr>
</table>

</body>
</html>"""

print(f'✅ HTML generado — {len(html):,} caracteres')

In [ ]:
# ── 10. PREVIEW EN COLAB ─────────────────────────────────────────────────────
from IPython.display import HTML
HTML(html)

In [ ]:
# ── 11. ENVÍO POR CORREO ──────────────────────────────────────────────────────
# Agrega/quita destinatarios según el trimestre
lista_correos = [
    'nelson.amparan@academiaglobal.mx',
    'karla.garzon@academiaglobal.mx',
    'jazmin.garzon@academiaglobal.mx',
    'javier.cazarez@academiaglobal.mx',
    'rosario.verduzco@academiaglobal.mx',
    'rodolfo.castro@academiaglobal.mx',
    'elsie.garzon@academiaglobal.mx',
    'vanessa.fuentes@academiaglobal.mx',
    'andreyely.ronquillo@academiaglobal.mx',
    'fernanda.castro@academiaglobal.mx',
    'nery.perez@academiaglobal.mx',
    'ernesto.torres@academiaglobal.mx',
    'anabelen.avila@academiaglobal.mx',
    'erik.velasco@academiaglobal.mx',
    'claudio.pacheco@academiaglobal.mx',
    'almendra.navidad@academiaglobal.mx',
    'abdiel.gutierrez@academiaglobal.mx',
    'leticia.beltran@academiaglobal.mx',
    'fedra.llanes@academiaglobal.mx',
    'analaura.roman@academiaglobal.mx',
    'mayra.alvarez@academiaglobal.mx',
]

REMITENTE    = 'siimag@academiaglobal.mx'
APP_PASSWORD = 'oigp hpxu immp gsks'  # ← mueve esto a una variable de entorno
ASUNTO       = f'Pulso Académico SIIMAG · {PERIODO_LABEL}'

msg = MIMEMultipart('alternative')
msg['Subject'] = ASUNTO
msg['From']    = REMITENTE
msg['Bcc']     = ', '.join(lista_correos)
msg.attach(MIMEText(html, 'html'))

with smtplib.SMTP('smtp.gmail.com', 587) as server:
    server.starttls()
    server.login(REMITENTE, APP_PASSWORD)
    server.send_message(msg)

print(f'✅ Correo enviado a {len(lista_correos)} destinatarios')